# 04 · FF is the single-scan degeneracy of PF

This notebook makes the package's founding thesis concrete and runnable:
**FF-HEDM is PF-HEDM with exactly one scan position.** Same orchestrator,
same stages, same on-disk contract; the only difference is the number of
beam positions the sample is scanned through.

We:

1. Show the `ScanGeometry` abstraction: `ff()` ≡ `pf_uniform(n_scans=1)`.
2. Run the FF pipeline end-to-end through indexing and confirm the binned
   `Spots.bin` already carries the PF-shaped 10-column layout (col 9 =
   ScanNr) with every spot at scan 0 — the single-scan degeneracy made
   literal in the bytes on disk.

> **Runtime** ~1–1.5 min on CPU.

In [ ]:
import os
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')

import shutil
import subprocess
import sys
import tempfile
import time
from pathlib import Path

import numpy as np

from midas_pipeline import __version__, ScanGeometry
print('midas-pipeline', __version__)

## 1. The geometry: FF is `pf_uniform` with one scan position

`ScanGeometry` is the single abstraction the orchestrator dispatches on.
`ScanGeometry.ff()` is constructed as a PF uniform scan with `n_scans=1`
at Y = 0 — there is no separate FF geometry type.

In [ ]:
ff = ScanGeometry.ff()
pf = ScanGeometry.pf_uniform(n_scans=5, scan_step_um=2.0, beam_size_um=4.0)
print('FF :', ff)
print('PF :', pf)
print()
print('FF scan_mode    :', ff.scan_mode)
print('FF n_scans      :', ff.n_scans, '(single position)')
print('FF positions    :', ff.scan_positions)
print('PF n_scans      :', pf.n_scans)
print('PF positions    :', pf.scan_positions, 'um')
assert ff.n_scans == 1 and float(ff.scan_positions[0]) == 0.0

## 2. Run FF end-to-end and inspect the on-disk degeneracy

The binning stage emits a **unified** `Spots.bin`: 10 columns where col 9
is `ScanNr`, plus an `int64`-pair `Data.bin` carrying `(spot_id, scan_nr)`
and a `positions.csv` sidecar. This is the *same* layout a PF run writes —
FF just has one scan position, so every `ScanNr` is 0 and `positions.csv`
is a single `0.0` line. The indexer reads this identical contract in
both modes.

In [ ]:
from midas_ff_pipeline.testing import generate_synthetic_dataset

MIDAS_HOME = Path(os.environ.get('MIDAS_HOME') or Path.home() / 'opt' / 'MIDAS')
PARAMS_TEMPLATE = MIDAS_HOME / 'FF_HEDM' / 'Example' / 'Parameters.txt'

WORK = Path(tempfile.mkdtemp(prefix='midas_pipeline_deg_'))
zarr = generate_synthetic_dataset(
    out_dir=WORK / 'sim', params_template=PARAMS_TEMPLATE,
    n_grains=20, n_cpus=4,
)
RESULT = WORK / 'run'
cmd = [
    sys.executable, '-m', 'midas_pipeline', 'run', '--scan-mode', 'ff',
    '--params', str(WORK / 'sim' / PARAMS_TEMPLATE.name),
    '--result', str(RESULT), '--zarr', str(zarr),
    '--n-cpus', '4', '--device', 'cpu', '--dtype', 'float64',
    '--indexer-backend', 'python',
    '--skip', 'refinement', '--skip', 'process_grains', '--skip', 'consolidation',
]
t0 = time.time()
proc = subprocess.run(cmd, capture_output=True, text=True)
print(f'exit={proc.returncode} ({time.time() - t0:.1f}s)')
assert proc.returncode == 0, proc.stderr[-2000:]

In [ ]:
layer = RESULT / 'LayerNr_1'

# Spots.bin: PF-shaped 10 columns; col 9 = ScanNr, all zero for FF.
spots = np.fromfile(layer / 'Spots.bin', dtype=np.float64).reshape(-1, 10)
print(f'Spots.bin       : {spots.shape[0]} spots x {spots.shape[1]} cols')
print(f'ScanNr (col 9)  : unique = {np.unique(spots[:, 9])}  -> single-scan degeneracy')

# Data.bin: int64 (spot_id, scan_nr) pairs; scan_nr column all zero.
data = np.fromfile(layer / 'Data.bin', dtype=np.int64).reshape(-1, 2)
print(f'Data.bin        : {data.shape[0]} (spot_id, scan_nr) entries; '
      f'scan_nr unique = {np.unique(data[:, 1])}')

# positions.csv: a single 0.0 line.
pos = (layer / 'positions.csv').read_text().split()
print(f'positions.csv   : {pos}  -> one scan position at Y=0')

assert np.all(spots[:, 9] == 0.0)
assert np.all(data[:, 1] == 0)
assert len(pos) == 1 and float(pos[0]) == 0.0

In [ ]:
# And the indexer recovered grains over this single-scan layout.
ib = np.fromfile(layer / 'IndexBest.bin', dtype=np.float64).reshape(-1, 15)
print(f'indexed seeds with a solution: {int((ib[:, 14] > 0).sum())} / {ib.shape[0]}')
assert int((ib[:, 14] > 0).sum()) > 0

## Takeaway

The FF run produced a `Spots.bin` / `Data.bin` / `positions.csv` triple
that is **structurally a PF run with `n_scans = 1`**: the ScanNr column
exists and is uniformly 0, and there is one scan position. Adding more
scan positions (PF) populates ScanNr `> 0` and a multi-row
`positions.csv`, unlocking the per-voxel reconstruction and V-map
refinement (notebook 03) — but the code path, file format, and indexer
are the same. That is the single-orchestrator, single-contract design the
package is built around.

In [ ]:
shutil.rmtree(WORK, ignore_errors=True)
print('cleaned', WORK)